# Spark: объединения и оконные вычисления


Показать проверку ключей, сохранение количества строк и расчёты внутри группы.

Используйте отдельное окно и отдельный контекст в каждой ячейке.

Ограничения: Когда нужно объединить справочник с фактами или добавить место, итог и соседнее значение без потери строк. Проверяйте кратность ключей и случай с пропусками; не используйте окно без устойчивого порядка.


## Основные понятия
Объединение сопоставляет строки по ключу. Оконное вычисление добавляет метрику к строкам, не сворачивая их. Перед объединением важно определить уровень детализации и кратность ключа.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Примеры


**1. Левое объединение**

Сохраните все строки игроков и добавьте название героя.



In [ ]:
with SparkSessionContext('example-join-left') as spark:
    players = spark.table("lakehouse.raw.pg__match_players").select("match_id", "hero_id", "kills")
    heroes = spark.table("lakehouse.raw.pg__heroes").select("hero_id", "localized_name")
    joined = players.join(heroes, "hero_id", "left")

    print(f"Строки до: {players.count()}, после: {joined.count()}")
    joined.select("match_id", "localized_name", "kills").show(10, truncate=False)

**2. Проверка кратности**

Найдите ключи справочника, которые могут размножить строки при объединении.



In [ ]:
with SparkSessionContext('example-join-cardinality') as spark:
    (
        spark
            .table("lakehouse.raw.pg__heroes")
            .groupBy("hero_id").count()
            .filter(F.col("count") > 1)
            .show(20, truncate=False)
    )

**3. Рейтинг внутри матча**

Присвойте игрокам места по числу убийств в каждом матче.



In [ ]:
with SparkSessionContext('example-window-rank') as spark:
    ordering = Window.partitionBy("match_id").orderBy(F.desc_nulls_last("kills"), F.asc("player_slot"))
    (
        spark
            .table("lakehouse.raw.pg__match_players")
            .select("match_id", "player_slot", "hero_id", "kills")
            .withColumn("place", F.row_number().over(ordering))
            .filter("place <= 3")
            .orderBy("match_id", "place")
            .show(30, truncate=False)
    )

**4. Нарастающий итог**

Посчитайте накопленное изменение золота по игроку и времени.



In [ ]:
with SparkSessionContext('example-window-running') as spark:
    ordering = Window.partitionBy("match_id", "player_slot").orderBy("time").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    (
        spark
            .table("lakehouse.raw.mongo__gold_events")
            .select("match_id", "player_slot", "time", "gold")
            .withColumn("running_gold", F.sum("gold").over(ordering))
            .orderBy("match_id", "player_slot", "time")
            .show(20, truncate=False)
    )

**5. Предыдущее значение**

Сравните событие с предыдущим событием того же игрока.



In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-window-lag') as spark:
    ordering = Window.partitionBy("match_id", "player_slot").orderBy("time")
    (
        spark
            .table("lakehouse.raw.mongo__gold_events")
            .select("match_id", "player_slot", "time", "gold")
            .withColumn("previous_gold", F.lag("gold").over(ordering))
            .limit(30)
            .show(truncate=False)
        )